# RUL Estimation — CNC Machining Centre (v3-fast, Colab-ready)

Leakage-corrected pipeline with a fair, budget-matched comparison of Decision Tree, Random Forest,
Gradient Boosting, LSTM and CNN-LSTM.

**Run in Colab:** Runtime → Change runtime type → **T4 GPU** → upload the CSV → Runtime → Run all.

**Speed settings** (top of Section 1):
- `FAST = True` (default): light Optuna search (8 trials), 3 final seeds. Roughly 20–30 min on a T4.
- `FAST = False`: full search (25 trials), 5 seeds. Expect 60–90 min.
- `RUN_TUNING = False`: skips the search and uses the stored parameters. Roughly 8–10 min.

Results are whatever the held-out machines produce; no target is forced.

In [ ]:
# ---- Settings ----
FAST = True            # False = full search (slower, more trials/seeds)
RUN_TUNING = True      # False = use stored best parameters (fastest)
N_TRIALS = 8 if FAST else 25
TUNE_EPOCHS = 15 if FAST else 25
FINAL_EPOCHS = 40 if FAST else 80
SEEDS = [42, 43, 44] if FAST else [42, 43, 44, 45, 46]
DATA_PATH = "cnc_machining_centre_rul_dataset__2_.csv"

%pip install -q optuna
import os, json, random, time, warnings
import numpy as np, pandas as pd
import matplotlib.pyplot as plt, seaborn as sns
from sklearn.preprocessing import StandardScaler
from sklearn.model_selection import GroupShuffleSplit
from sklearn.tree import DecisionTreeRegressor
from sklearn.ensemble import RandomForestRegressor, HistGradientBoostingRegressor
from sklearn.metrics import mean_squared_error, mean_absolute_error, r2_score
import tensorflow as tf
from tensorflow import keras
from tensorflow.keras import layers, Sequential
from tensorflow.keras.callbacks import EarlyStopping, ReduceLROnPlateau
import optuna

warnings.filterwarnings("ignore")
optuna.logging.set_verbosity(optuna.logging.WARNING)
sns.set_theme(style="whitegrid")
SEED = 42
def set_seed(s):
    random.seed(s); np.random.seed(s); tf.random.set_seed(s)
set_seed(SEED)
print("TensorFlow", tf.__version__, "| GPU:", bool(tf.config.list_physical_devices("GPU")))
T0 = time.time()

## 2. Load, Dedupe, Column Roles

In [ ]:
try:
    from google.colab import files
    if not os.path.exists(DATA_PATH):
        DATA_PATH = list(files.upload())[0]
except ImportError:
    pass

df = pd.read_csv(DATA_PATH).drop_duplicates().sort_values(["unit_id", "cycle"]).reset_index(drop=True)
print("Rows:", len(df), "| machines:", df.unit_id.nunique())

ID_COLS = ["unit_id", "cycle", "timestamp", "shift", "machine_type"]
TARGET = "RUL"
LEAKAGE_COLS = ["RUL", "label_binary_30", "label_binary_15", "health_state", "failure_mode"]
DISTRACTORS = ["ambient_light_lux", "power_grid_voltage_V"]
SENSOR_COLS = [c for c in df.columns if c not in ID_COLS + LEAKAGE_COLS + DISTRACTORS]
assert not set(SENSOR_COLS) & set(LEAKAGE_COLS)
print(len(SENSOR_COLS), "sensor features")

## 3. Split by Machine, Clean (training statistics only), Engineer Causal Features

In [ ]:
gss = GroupShuffleSplit(n_splits=1, test_size=0.2, random_state=SEED)
tr_idx, te_idx = next(gss.split(df, groups=df["unit_id"]))
train = df.iloc[tr_idx].copy(); test = df.iloc[te_idx].copy()
assert set(train.unit_id).isdisjoint(test.unit_id)

def ffill_machine(part):
    part = part.copy()
    part[SENSOR_COLS] = part.groupby("unit_id")[SENSOR_COLS].transform(lambda s: s.ffill())
    return part
train, test = ffill_machine(train), ffill_machine(test)

med = train[SENSOR_COLS].median()
q1, q3 = train[SENSOR_COLS].quantile(0.25), train[SENSOR_COLS].quantile(0.75)
lo, hi = q1 - 3 * (q3 - q1), q3 + 3 * (q3 - q1)
for p in (train, test):
    p[SENSOR_COLS] = p[SENSOR_COLS].fillna(med).clip(lower=lo, upper=hi, axis=1)

KEY = [c for c in ["spindle_vibration_x_mm_s", "spindle_vibration_y_mm_s", "spindle_vibration_z_mm_s",
                   "bearing_temperature_C", "motor_current_A", "spindle_torque_Nm", "tool_wear_mm",
                   "acoustic_emission_dB", "cutting_force_N", "power_consumption_kW"] if c in SENSOR_COLS]

def add_features(part):
    g = part.groupby("unit_id"); new = {}
    for c in KEY:
        new[f"{c}_mean5"] = g[c].transform(lambda s: s.rolling(5, min_periods=1).mean())
        new[f"{c}_std5"] = g[c].transform(lambda s: s.rolling(5, min_periods=2).std().fillna(0))
        new[f"{c}_mean15"] = g[c].transform(lambda s: s.rolling(15, min_periods=1).mean())
        new[f"{c}_diff1"] = g[c].diff().fillna(0)
        new[f"{c}_ewm10"] = g[c].transform(lambda s: s.ewm(span=10, adjust=False).mean())
    return pd.concat([part, pd.DataFrame(new, index=part.index)], axis=1)

train, test = add_features(train), add_features(test)
FEATURES = SENSOR_COLS + [c for c in train.columns if c not in df.columns]

scaler = StandardScaler().fit(train[FEATURES])
train[FEATURES] = scaler.transform(train[FEATURES])
test[FEATURES] = scaler.transform(test[FEATURES])
RUL_CLIP = 130
print("Features:", len(FEATURES), "| train rows:", len(train), "| test rows:", len(test))

## 4. Metrics

In [ ]:
def asym_score(y, p):
    d = np.asarray(p, float) - np.asarray(y, float)
    return float(np.sum(np.where(d < 0, np.exp(-d / 13) - 1, np.exp(d / 10) - 1)))

def metrics(y, p):
    return {"RMSE": mean_squared_error(y, p) ** 0.5, "MAE": mean_absolute_error(y, p),
            "R2": r2_score(y, p), "AsymScore": asym_score(y, p)}

X_tr, y_tr = train[FEATURES].values, train[TARGET].clip(upper=RUL_CLIP).values
X_te, y_te = test[FEATURES].values, test[TARGET].clip(upper=RUL_CLIP).values

## 5. Tabular Models (Decision Tree, Random Forest, Gradient Boosting)

In [ ]:
results, preds = [], {}
TAB_MODELS = {}
def run_tabular(name, make):
    m = make().fit(X_tr, y_tr)
    TAB_MODELS[name] = m
    p = m.predict(X_te)
    row = {"Model": name, **metrics(y_te, p)}
    results.append(row); preds[name] = p
    print(f"{name:>20s} | R2 {row['R2']:.3f} | RMSE {row['RMSE']:.2f} | MAE {row['MAE']:.2f}")

run_tabular("Decision Tree", lambda: DecisionTreeRegressor(max_depth=8, min_samples_leaf=15, random_state=SEED))
run_tabular("Random Forest", lambda: RandomForestRegressor(n_estimators=120, max_depth=14, min_samples_leaf=5,
                                                           n_jobs=-1, random_state=SEED))
run_tabular("Gradient Boosting", lambda: HistGradientBoostingRegressor(max_iter=300, learning_rate=0.06,
                                                                       random_state=SEED))

## 6. Sequence Builder and Architectures

In [ ]:
def make_sequences(part, window):
    X, y = [], []
    for _, g in part.groupby("unit_id", sort=False):
        F = g[FEATURES].values.astype(np.float32)
        r = g[TARGET].clip(upper=RUL_CLIP).values.astype(np.float32)
        Fp = np.vstack([np.zeros((window - 1, F.shape[1]), np.float32), F])
        for i in range(len(F)):
            X.append(Fp[i:i + window]); y.append(r[i])
    return np.stack(X), np.array(y, np.float32)

# Inner machine-level split of the TRAINING data: used for tuning and early stopping only
a, b = next(GroupShuffleSplit(n_splits=1, test_size=0.2, random_state=SEED + 1).split(train, groups=train.unit_id))
inner_tr, inner_va = train.iloc[a], train.iloc[b]
_cache = {}
def inner_seq(w):
    if w not in _cache:
        _cache[w] = (*make_sequences(inner_tr, w), *make_sequences(inner_va, w))
    return _cache[w]

def build_lstm(w, nf, u1, u2, do, lr):
    m = Sequential([layers.Input((w, nf)), layers.LSTM(u1, return_sequences=True), layers.Dropout(do),
                    layers.LSTM(u2), layers.Dropout(do), layers.Dense(32, activation="relu"), layers.Dense(1)])
    m.compile(keras.optimizers.Adam(lr), loss="mse"); return m

def build_cnn_lstm(w, nf, filters, k, u1, u2, do, lr):
    m = Sequential([layers.Input((w, nf)),
                    layers.Conv1D(filters, k, padding="same", activation="relu"), layers.BatchNormalization(),
                    layers.Conv1D(filters // 2, k, padding="same", activation="relu"),
                    layers.MaxPooling1D(2, padding="same"),
                    layers.LSTM(u1, return_sequences=True), layers.Dropout(do),
                    layers.LSTM(u2), layers.Dropout(do), layers.Dense(32, activation="relu"), layers.Dense(1)])
    m.compile(keras.optimizers.Adam(lr), loss="mse"); return m

def fit_seq(make, Xa, ya, Xb, yb, epochs, batch, seed):
    set_seed(seed); m = make()
    m.fit(Xa, ya, validation_data=(Xb, yb), epochs=epochs, batch_size=batch, verbose=0,
          callbacks=[EarlyStopping("val_loss", patience=6, restore_best_weights=True),
                     ReduceLROnPlateau("val_loss", factor=0.5, patience=3, min_lr=1e-5)])
    return m

## 7. Hyperparameter Search (Optuna, budget-matched for both deep models)
Both models get the same number of trials. The search space is kept small on purpose so it finishes in
reasonable Colab time. Set `RUN_TUNING = False` in Section 1 to skip it and use the stored parameters.

In [ ]:
DEFAULT_CNN = dict(window=30, filters=64, kernel=3, lstm1=64, lstm2=32, dropout=0.2, lr=1e-3, batch=256)
DEFAULT_LSTM = dict(window=30, units1=64, units2=32, dropout=0.2, lr=1e-3, batch=256)

def obj_cnn(trial):
    w = trial.suggest_categorical("window", [20, 30])
    Xa, ya, Xb, yb = inner_seq(w)
    f = trial.suggest_categorical("filters", [32, 64])
    k = trial.suggest_categorical("kernel", [3, 5])
    u1 = trial.suggest_categorical("lstm1", [32, 64])
    u2 = trial.suggest_categorical("lstm2", [16, 32])
    do = trial.suggest_float("dropout", 0.1, 0.3)
    lr = trial.suggest_float("lr", 3e-4, 3e-3, log=True)
    bs = trial.suggest_categorical("batch", [256, 512])
    m = fit_seq(lambda: build_cnn_lstm(w, Xa.shape[2], f, k, u1, u2, do, lr), Xa, ya, Xb, yb, TUNE_EPOCHS, bs, SEED)
    return float(np.sqrt(mean_squared_error(yb, m.predict(Xb, verbose=0).ravel())))

def obj_lstm(trial):
    w = trial.suggest_categorical("window", [20, 30])
    Xa, ya, Xb, yb = inner_seq(w)
    u1 = trial.suggest_categorical("units1", [32, 64])
    u2 = trial.suggest_categorical("units2", [16, 32])
    do = trial.suggest_float("dropout", 0.1, 0.3)
    lr = trial.suggest_float("lr", 3e-4, 3e-3, log=True)
    bs = trial.suggest_categorical("batch", [256, 512])
    m = fit_seq(lambda: build_lstm(w, Xa.shape[2], u1, u2, do, lr), Xa, ya, Xb, yb, TUNE_EPOCHS, bs, SEED)
    return float(np.sqrt(mean_squared_error(yb, m.predict(Xb, verbose=0).ravel())))

if RUN_TUNING:
    t = time.time()
    s1 = optuna.create_study(direction="minimize", sampler=optuna.samplers.TPESampler(seed=SEED))
    s1.optimize(obj_cnn, n_trials=N_TRIALS)
    s2 = optuna.create_study(direction="minimize", sampler=optuna.samplers.TPESampler(seed=SEED))
    s2.optimize(obj_lstm, n_trials=N_TRIALS)
    best_cnn, best_lstm = s1.best_params, s2.best_params
    print(f"Tuning took {(time.time()-t)/60:.1f} min")
    print("CNN-LSTM best (val RMSE %.3f):" % s1.best_value, best_cnn)
    print("LSTM best (val RMSE %.3f):" % s2.best_value, best_lstm)
else:
    best_cnn, best_lstm = DEFAULT_CNN, DEFAULT_LSTM
    print("Using stored parameters")

os.makedirs("outputs", exist_ok=True)
json.dump({"cnn_lstm": best_cnn, "lstm": best_lstm}, open("outputs/best_params.json", "w"), indent=2)

## 8. Final Training and Test Evaluation (multi-seed)

In [ ]:
FINAL_MODELS = {}
def run_final(name, make_fn, w, bs):
    Xa, ya, Xb, yb = inner_seq(w)
    Xte, yte = make_sequences(test, w)
    assert np.allclose(yte, y_te, atol=1e-4)
    rows, plist, last = [], [], None
    for s in SEEDS:
        last = fit_seq(make_fn, Xa, ya, Xb, yb, FINAL_EPOCHS, bs, s)
        FINAL_MODELS.setdefault(name, []).append(last)
        p = last.predict(Xte, verbose=0).ravel()
        plist.append(p); rows.append(metrics(yte, p))
    summ = pd.DataFrame(rows)
    print(f"{name:>20s} | R2 {summ.R2.mean():.3f}±{summ.R2.std():.3f} | "
          f"RMSE {summ.RMSE.mean():.2f}±{summ.RMSE.std():.2f} | MAE {summ.MAE.mean():.2f}")
    return {"Model": name, **{k: summ[k].mean() for k in summ}, **{k + "_std": summ[k].std() for k in summ}}, \
           np.mean(plist, axis=0), last

nf = len(FEATURES)
t = time.time()
rl, pl, _ = run_final("LSTM (tuned)",
    lambda: build_lstm(best_lstm["window"], nf, best_lstm["units1"], best_lstm["units2"],
                       best_lstm["dropout"], best_lstm["lr"]),
    best_lstm["window"], best_lstm["batch"])
rc, pc, model_cnn = run_final("CNN-LSTM (tuned)",
    lambda: build_cnn_lstm(best_cnn["window"], nf, best_cnn["filters"], best_cnn["kernel"],
                           best_cnn["lstm1"], best_cnn["lstm2"], best_cnn["dropout"], best_cnn["lr"]),
    best_cnn["window"], best_cnn["batch"])
print(f"Final training took {(time.time()-t)/60:.1f} min")
results += [rl, rc]; preds["LSTM (tuned)"] = pl; preds["CNN-LSTM (tuned)"] = pc

## 9. Comparison, Plots, Saved Outputs

In [ ]:
res = pd.DataFrame(results)[["Model", "R2", "R2_std", "RMSE", "RMSE_std", "MAE", "AsymScore"]]
res = res.sort_values("R2", ascending=False).reset_index(drop=True)
print(res.round(3).to_string(index=False))

fig, ax = plt.subplots(1, 2, figsize=(14, 5))
sns.barplot(data=res, x="Model", y="R2", ax=ax[0], palette="crest"); ax[0].set_title("Test R² (higher is better)")
sns.barplot(data=res, x="Model", y="RMSE", ax=ax[1], palette="magma"); ax[1].set_title("Test RMSE (lower is better)")
for a_ in ax: a_.tick_params(axis="x", rotation=25)
plt.tight_layout(); plt.show()

uid = test["unit_id"].iloc[0]; mk = (test["unit_id"] == uid).values
plt.figure(figsize=(11, 5)); plt.plot(test.loc[mk, "cycle"], y_te[mk], "k-", lw=2.5, label="Actual")
for n, p in preds.items(): plt.plot(test.loc[mk, "cycle"], p[mk], "--", label=n)
plt.xlabel("Cycle"); plt.ylabel("RUL (clipped)"); plt.title(f"Held-out machine #{uid}")
plt.legend(); plt.tight_layout(); plt.show()

res.to_csv("outputs/model_comparison.csv", index=False)
out = test[["unit_id", "cycle", TARGET]].reset_index(drop=True); out["RUL_clipped"] = y_te
for n, p in preds.items(): out["pred_" + n.replace(" ", "_").replace("(", "").replace(")", "")] = p
out.to_csv("outputs/test_predictions.csv", index=False)
model_cnn.save("outputs/cnn_lstm_final.keras")
print(f"Total runtime: {(time.time()-T0)/60:.1f} min | files saved to ./outputs/")

In [ ]:

import joblib
from pathlib import Path
EXPORT = Path("outputs/models"); EXPORT.mkdir(parents=True, exist_ok=True)

def slug(name):
    return name.split(" (")[0].lower().replace("-", "_").replace(" ", "_")

model_entries = []
for name, m in TAB_MODELS.items():
    fname = slug(name) + ".joblib"
    joblib.dump(m, EXPORT / fname)
    model_entries.append({"name": name, "kind": "tabular", "files": [fname], "window": None})

for name, ms in FINAL_MODELS.items():
    window = best_cnn["window"] if "CNN" in name else best_lstm["window"]
    files = []
    for i, m in enumerate(ms):
        fn = f"{slug(name)}_seed{SEEDS[i]}.keras"
        m.save(EXPORT / fn); files.append(fn)
    model_entries.append({"name": name, "kind": "deep", "files": files, "window": int(window)})

config = {
    "format_version": 1,
    "sensor_cols": SENSOR_COLS,
    "key_sensors": KEY,
    "features": FEATURES,
    "rul_clip": RUL_CLIP,
    "dedupe": True,
    "median": {c: float(med[c]) for c in SENSOR_COLS},
    "lower": {c: float(lo[c]) for c in SENSOR_COLS},
    "upper": {c: float(hi[c]) for c in SENSOR_COLS},
    "scaler_mean": [float(x) for x in scaler.mean_],
    "scaler_scale": [float(x) for x in scaler.scale_],
    "models": model_entries,
    "sklearn_version": __import__("sklearn").__version__,
    "tensorflow_version": tf.__version__,
}
with open(EXPORT / "preprocessing.json", "w") as f:
    json.dump(config, f, indent=2)
print("Exported", len(model_entries), "model entries to outputs/models/")
for e in model_entries: print(" -", e["name"], e["kind"], e["files"])

## 10. Notes for the Dissertation
- Report the ranking from **your** run (Section 9), not from expectations.
- If the CNN-LSTM–LSTM gap is smaller than the seed standard deviation, say the models are statistically
  comparable rather than claiming superiority.
- `FAST = True` uses fewer trials and seeds than the full protocol. State which setting produced the reported numbers.
- The dataset is synthetic; generalisation to real plant data is not yet demonstrated.